# Calculate metrics: treatment-main LRT + treatment pattern + supplementary entropy

Runs after the primary treatment-effect screen `02_drimseq_treatment_main_pseudobulk.R` (full = `~ bs(pt) + super_batch + induced`, null = `~ bs(pt) + super_batch`, single LRT per gene). Per lineage, this notebook:

1. Reads the per-gene main-effect LRT output (`drimseq_gene_<lineage>_treatment_main.csv`) and renames its test columns to `*_treatment` for clarity.
2. Classifies each gene by sign(`delta_psi_treatment`) into `treatment_lengthening`, `treatment_shortening`, or `non_significant`, using `padj_treatment < 0.05`.
3. Computes per-pseudobulk-unit Shannon entropy of UTR-site usage, aggregates to per-sample (cluster-aware) and then per-condition, and tests delta_entropy with a two-sided Wilcoxon rank-sum over 14 vs 8 sample-level values. BH-corrected over genes. Supplementary; not used in the composite ranking.
4. Writes `metrics_per_gene.csv`, `per_unit_entropy.csv`, and `per_sample_entropy.csv` per lineage.

Statistical notes:
- `padj_treatment` is already BH-adjusted on the post-filter gene set by DRIMSeq; we do not re-correct.
- Per-feature p-values from the feature_diagnostic CSV are not used; the gene-level LRT is the only inferentially valid p-value (compositional non-independence).
- Entropy uses raw pseudobulk empirical proportions, not DRIMSeq-fitted ones, i.e. the same data the LRT tests.
- Sample-level Wilcoxon is the cluster-aware test (sample = unit of replication for between-condition contrasts).


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "environment" / "apa-python.yml").is_file()
    )
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/organoids_knoblich/UTR_treatment")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import os, time, warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix, issparse
from scipy.stats import ranksums, spearmanr
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "savefig.dpi": 300,
        "font.size": 7,
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "axes.titlesize": 8,
        "axes.labelsize": 7,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.linewidth": 0.8,
    }
)
sc.settings.verbosity = 1
print("Setup complete.")

In [ ]:
LINEAGES = ["inhibitory", "excitatory"]
N_BINS = 8
PADJ_THRESHOLD = 0.05

BASEPATH = Path(f"{DATA_ROOT}/3utr/organoids/knoblich")
UTR_DIR = BASEPATH / "utrcalling"
RESULTS_ROOT = UTR_DIR / "dirmult_treatment_results"
H5AD_PATH = UTR_DIR / "merged_bw12_above10cts_apa_named_annotated_fix_pt.h5ad"


def lineage_dir(lineage):
    return RESULTS_ROOT / lineage / "main_only"


def out_dir(lineage):
    p = RESULTS_ROOT / lineage / "calculate_metrics"
    p.mkdir(parents=True, exist_ok=True)
    return p


assert H5AD_PATH.exists(), f"missing {H5AD_PATH}"
for ln in LINEAGES:
    assert lineage_dir(ln).exists(), f"missing {lineage_dir(ln)}"
    print(f"  {ln:12s}  in={lineage_dir(ln)}\n               out={out_dir(ln)}")

## Step 1: load the main-effect LRT table and classify treatment pattern

In [ ]:
def load_lrt_table(lineage):
    df = pd.read_csv(lineage_dir(lineage) / f"drimseq_gene_{lineage}_treatment_main.csv")
    # Rename the test columns so downstream code reads `*_treatment`
    df = df.rename(
        columns={
            "lr": "lr_treatment",
            "df": "df_treatment",
            "pvalue": "pvalue_treatment",
            "adj_pvalue": "padj_treatment",
        }
    )
    # Drop the constant `test_type` column to avoid downstream confusion
    if "test_type" in df.columns:
        df = df.drop(columns=["test_type"])
    return df


lrt = {ln: load_lrt_table(ln) for ln in LINEAGES}
for ln, df in lrt.items():
    sig05 = (df["padj_treatment"] < PADJ_THRESHOLD).sum()
    sig01 = (df["padj_treatment"] < 0.01).sum()
    n_nan = df["padj_treatment"].isna().sum()
    print(
        f"[{ln}]  n_genes={len(df):,}  NaN padj: {n_nan}  "
        f"padj<{PADJ_THRESHOLD}: {sig05}  padj<0.01: {sig01}"
    )
lrt["inhibitory"].head(3)

In [ ]:
def classify_pattern(row, padj_th=PADJ_THRESHOLD):
    if pd.isna(row["padj_treatment"]) or row["padj_treatment"] >= padj_th:
        return "non_significant"
    if pd.isna(row["delta_psi_treatment"]):
        return "non_significant"
    return "treatment_lengthening" if row["delta_psi_treatment"] > 0 else "treatment_shortening"


for ln in LINEAGES:
    lrt[ln]["pattern"] = lrt[ln].apply(classify_pattern, axis=1)
    print(f"\n[{ln}] pattern counts:")
    print(lrt[ln]["pattern"].value_counts().to_string())

## Step 2: per-pseudobulk-unit Shannon entropy

For each retained pseudobulk unit $u$ (from `pseudobulk_units.csv`) and each post-filter gene $g$:

$$H_g(u) = -\sum_{k=1}^{K_g} \pi_{uk} \log_2 \pi_{uk} \,/\, \log_2 K_g \in [0, 1]$$

We use the same quantile binning and unit filtering as the R script and cross-check per-unit `n_cells` against the manifest before computing entropy.

In [ ]:
print("Loading h5ad ...")
t0 = time.time()
adata = sc.read_h5ad(H5AD_PATH)
print(f"  loaded in {time.time() - t0:.1f}s  shape={adata.shape}")
adata.obs["induced_str"] = adata.obs["induced"].astype(str).str.upper()

In [ ]:
REQUIRE_TARGETING_GUIDE = (
    True  # cleaned pooled: treatment = targeting guides only (drop induced Control2/NaN)
)


def pseudobulk_lineage(adata, lineage, units_df, n_bins=N_BINS):
    pt_col = f"dpt_pseudotime_{lineage}"
    obs = adata.obs
    cell_idx_all = np.where(obs[pt_col].notna().values)[0]
    if REQUIRE_TARGETING_GUIDE and "gRNA" in obs.columns:
        _ind = obs["induced"].astype(str).str.upper().values[cell_idx_all]
        _g = obs["gRNA"].astype(str).values[cell_idx_all]
        _drop = (_ind == "TRUE") & ((_g == "Control2") | (_g == "nan"))
        cell_idx_all = cell_idx_all[~_drop]
    pt = obs[pt_col].values[cell_idx_all].astype(float)
    sample_id = obs["sample_id"].astype(str).values[cell_idx_all]
    qs = np.unique(np.quantile(pt, np.linspace(0, 1, n_bins + 1)))
    # R `cut(include.lowest=TRUE)` semantics: right-closed intervals
    bin_id = np.searchsorted(qs[1:-1], pt, side="left") + 1
    bin_id = np.clip(bin_id, 1, len(qs) - 1)
    unit_key = np.array([f"{s}__bin{b:02d}" for s, b in zip(sample_id, bin_id)])
    keep_units = set(units_df["sample_id"].astype(str))
    in_kept = np.isin(unit_key, list(keep_units))
    cell_idx_kept = cell_idx_all[in_kept]
    unit_key_kept = unit_key[in_kept]
    unique_units = np.array(sorted(set(unit_key_kept)))
    n_units = len(unique_units)
    unit_to_col = {u: i for i, u in enumerate(unique_units)}
    cols = np.array([unit_to_col[u] for u in unit_key_kept])
    rows = np.arange(len(cell_idx_kept))
    M = csr_matrix(
        (np.ones(len(cell_idx_kept), dtype=np.float64), (rows, cols)),
        shape=(len(cell_idx_kept), n_units),
    )
    X = adata.X[cell_idx_kept]
    if not issparse(X):
        X = csr_matrix(X)
    pb_counts = (X.T @ M).toarray().astype(np.float64)
    n_cells_per_unit = np.bincount(cols, minlength=n_units)
    pb_meta = pd.DataFrame({"sample_id": unique_units, "n_cells_recomputed": n_cells_per_unit})
    pb_meta = pb_meta.merge(units_df, on="sample_id", how="left")
    mismatch = int((pb_meta["n_cells_recomputed"] != pb_meta["n_cells"]).sum())
    if mismatch:
        print(f"  WARNING: {mismatch}/{n_units} units mismatch manifest n_cells")
    else:
        print(f"  re-pseudobulk OK: all {n_units} units match manifest n_cells")
    return pb_counts, pb_meta, unique_units


def compute_per_unit_entropy(pb_counts, unique_units, post_filter_feat_df, adata):
    feat_id_to_idx = {f: i for i, f in enumerate(adata.var_names)}
    gene_to_feat_idxs = defaultdict(list)
    missing = 0
    for _, r in post_filter_feat_df.iterrows():
        if r["feature_id"] not in feat_id_to_idx:
            missing += 1
            continue
        gene_to_feat_idxs[r["gene_id"]].append(feat_id_to_idx[r["feature_id"]])
    if missing:
        print(f"  WARNING: {missing} feature_ids not found in adata.var_names")
    records = []
    for gene_id, idxs in gene_to_feat_idxs.items():
        K = len(idxs)
        if K < 2:
            continue
        sub = pb_counts[idxs, :]
        totals = sub.sum(axis=0)
        valid = totals > 0
        H = np.full(pb_counts.shape[1], np.nan)
        if valid.any():
            pi = sub[:, valid] / totals[valid]
            with np.errstate(divide="ignore", invalid="ignore"):
                log2p = np.where(pi > 0, np.log2(pi), 0.0)
            H[valid] = -np.sum(pi * log2p, axis=0) / np.log2(K)
        for u_idx, u_key in enumerate(unique_units):
            records.append((gene_id, K, u_key, float(totals[u_idx]), float(H[u_idx])))
    return pd.DataFrame(records, columns=["gene_id", "K", "sample_id", "total_u", "H_g_u"])


per_unit_entropy_long = {}
for ln in LINEAGES:
    print(f"\n[{ln}] Re-pseudobulk + per-unit entropy ...")
    units_df = pd.read_csv(lineage_dir(ln) / "pseudobulk_units.csv")
    feat_df = pd.read_csv(lineage_dir(ln) / f"delta_psi_treatment_feature_{ln}.csv")
    pb_counts, pb_meta, unique_units = pseudobulk_lineage(adata, ln, units_df)
    df_h = compute_per_unit_entropy(pb_counts, unique_units, feat_df, adata)
    print(
        f"  per-unit entropy rows: {len(df_h):,} ({df_h['gene_id'].nunique():,} genes x {len(unique_units)} units)"
    )
    units_df["induced"] = units_df["induced"].astype(str).str.upper()
    df_h = df_h.merge(
        units_df[
            ["sample_id", "sample_id_root", "bin_id", "pt_mid", "induced", "super_batch", "n_cells"]
        ],
        on="sample_id",
        how="left",
    )
    per_unit_entropy_long[ln] = df_h
    print(f"  rows with NaN H (zero-total): {df_h['H_g_u'].isna().sum():,}")

## Step 3: sample-level aggregation + Wilcoxon (cluster-aware test of delta_entropy)

In [ ]:
def wilcoxon_per_gene(df_long):
    df = df_long.dropna(subset=["H_g_u"]).copy()
    df["w"] = df["n_cells"]
    df["wH"] = df["H_g_u"] * df["w"]
    per_sample = df.groupby(["gene_id", "sample_id_root"], as_index=False).agg(
        wH_sum=("wH", "sum"),
        w_sum=("w", "sum"),
        induced=("induced", "first"),
        super_batch=("super_batch", "first"),
    )
    per_sample["H_sample"] = per_sample["wH_sum"] / per_sample["w_sum"]
    per_sample = per_sample.drop(columns=["wH_sum", "w_sum"])
    rows = []
    for gene_id, g in per_sample.groupby("gene_id"):
        hT = g.loc[g["induced"] == "TRUE", "H_sample"].values
        hF = g.loc[g["induced"] == "FALSE", "H_sample"].values
        nT, nF = len(hT), len(hF)
        if nT < 2 or nF < 2:
            rows.append((gene_id, np.nan, np.nan, np.nan, np.nan, nT, nF))
            continue
        H_T = float(np.mean(hT))
        H_F = float(np.mean(hF))
        _, p = ranksums(hT, hF)
        rows.append((gene_id, H_T, H_F, H_T - H_F, float(p), nT, nF))
    out = pd.DataFrame(
        rows,
        columns=[
            "gene_id",
            "H_induced_True",
            "H_induced_False",
            "delta_entropy",
            "entropy_pvalue_wilcoxon",
            "n_samples_True_for_H",
            "n_samples_False_for_H",
        ],
    )
    valid = out["entropy_pvalue_wilcoxon"].notna()
    out["entropy_padj_bh"] = np.nan
    if valid.any():
        _, padj_v, _, _ = multipletests(
            out.loc[valid, "entropy_pvalue_wilcoxon"].values, method="fdr_bh"
        )
        out.loc[valid, "entropy_padj_bh"] = padj_v
    return out, per_sample


entropy_gene = {}
entropy_per_sample = {}
for ln in LINEAGES:
    print(f"\n[{ln}] sample-level Wilcoxon ...")
    out, ps = wilcoxon_per_gene(per_unit_entropy_long[ln])
    n_tested = int(out["entropy_pvalue_wilcoxon"].notna().sum())
    print(f"  genes tested: {n_tested:,}")
    print(f"  entropy_padj_bh < 0.05: {(out['entropy_padj_bh'] < 0.05).sum():,}")
    print(
        f"  delta_entropy range: [{out['delta_entropy'].min():.4f}, {out['delta_entropy'].max():.4f}]  "
        f"median={out['delta_entropy'].median():.4f}"
    )
    entropy_gene[ln] = out
    entropy_per_sample[ln] = ps

## Step 4: merge entropy onto LRT table and write outputs

In [ ]:
metrics = {}
for ln in LINEAGES:
    df = lrt[ln].merge(entropy_gene[ln], on="gene_id", how="left")
    metrics[ln] = df
    out_gene = out_dir(ln) / "metrics_per_gene.csv"
    out_unit = out_dir(ln) / "per_unit_entropy.csv"
    out_samp = out_dir(ln) / "per_sample_entropy.csv"
    df.to_csv(out_gene, index=False)
    per_unit_entropy_long[ln].to_csv(out_unit, index=False)
    entropy_per_sample[ln].to_csv(out_samp, index=False)
    print(f"[{ln}]")
    print(f"  metrics_per_gene.csv      {df.shape}  -> {out_gene}")
    print(f"  per_unit_entropy.csv      {per_unit_entropy_long[ln].shape}  -> {out_unit}")
    print(f"  per_sample_entropy.csv    {entropy_per_sample[ln].shape}  -> {out_samp}")
print("\nColumn order:", list(metrics["inhibitory"].columns))

## Diagnostics: drimseq_summary echo

In [ ]:
for ln in LINEAGES:
    s = pd.read_csv(lineage_dir(ln) / f"drimseq_summary_{ln}.csv")
    row = s.iloc[0]
    print(f"\n[{ln}]")
    for c in [
        "n_cells_lineage",
        "n_units_pseudobulk",
        "n_genes_pre_filter",
        "n_genes_post_filter",
        "n_features_pre_filter",
        "n_features_post_filter",
        "common_precision",
        "n_genes_precision_nan_repaired",
        "n_sig_gene_main_bh05",
        "n_sig_gene_main_bh01",
        "ks_uniformity_p_main",
        "bootstrap_b",
    ]:
        if c in s.columns:
            v = row[c]
            if isinstance(v, float) and (abs(v) > 1e6 or (0 < abs(v) < 1e-3)):
                print(f"  {c:35s} {v:.3e}")
            else:
                print(f"  {c:35s} {v}")

## Histograms: Δψ, ΔH, -log10(padj_treatment), pattern counts

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for row, ln in enumerate(LINEAGES):
    df = metrics[ln]
    ax = axes[row, 0]
    ax.hist(
        df["delta_psi_treatment"].dropna(),
        bins=60,
        color="#2980b9",
        edgecolor="black",
        linewidth=0.2,
    )
    ax.axvline(0, color="black", lw=0.6, ls="--")
    ax.set_title(f"{ln}: delta_psi_treatment")
    ax.set_xlabel("delta_psi_treatment")
    ax.set_ylabel("genes")

    ax = axes[row, 1]
    ax.hist(
        df["delta_entropy"].dropna(), bins=60, color="#16a085", edgecolor="black", linewidth=0.2
    )
    ax.axvline(0, color="black", lw=0.6, ls="--")
    ax.set_title(f"{ln}: delta_entropy (supplementary)")
    ax.set_xlabel("delta_entropy")

    ax = axes[row, 2]
    neglog = -np.log10(df["padj_treatment"].clip(lower=1e-300))
    ax.hist(neglog, bins=60, color="#c0392b", edgecolor="black", linewidth=0.2)
    ax.axvline(-np.log10(0.05), color="black", lw=0.6, ls="--", label="padj=0.05")
    ax.set_title(f"{ln}: -log10(padj_treatment)")
    ax.set_xlabel("-log10(padj)")
    ax.legend(fontsize=6)

    ax = axes[row, 3]
    pat_order = ["non_significant", "treatment_lengthening", "treatment_shortening"]
    counts = df["pattern"].value_counts().reindex(pat_order).fillna(0).astype(int)
    colors = ["#cccccc", "#1f77b4", "#d62728"]
    ax.bar(range(len(counts)), counts.values, color=colors)
    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels(counts.index, rotation=30, ha="right", fontsize=6)
    ax.set_title(f"{ln}: pattern counts")
    for i, v in enumerate(counts.values):
        ax.text(i, v, str(v), ha="center", va="bottom", fontsize=6)
plt.tight_layout()
plt.show()

## Δψ vs ΔH independence check

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, ln in zip(axes, LINEAGES):
    df = metrics[ln]
    m = df[["delta_psi_treatment", "delta_entropy"]].notna().all(axis=1)
    x = df.loc[m, "delta_psi_treatment"].values
    y = df.loc[m, "delta_entropy"].values
    sig = (df.loc[m, "padj_treatment"] < PADJ_THRESHOLD).values
    ax.scatter(
        x[~sig],
        y[~sig],
        s=3,
        alpha=0.25,
        c="#555555",
        rasterized=True,
        label=f"ns (n={(~sig).sum():,})",
    )
    ax.scatter(
        x[sig],
        y[sig],
        s=12,
        alpha=0.85,
        c="#e74c3c",
        rasterized=True,
        label=f"padj<{PADJ_THRESHOLD} (n={sig.sum()})",
    )
    rho, p = spearmanr(x, y)
    ax.axhline(0, color="grey", lw=0.5, ls="--")
    ax.axvline(0, color="grey", lw=0.5, ls="--")
    ax.set_xlabel("delta_psi_treatment")
    ax.set_ylabel("delta_entropy")
    ax.set_title(f"{ln}  Spearman={rho:.3f} (n={m.sum():,})")
    ax.legend(fontsize=6, frameon=False, loc="upper left")
plt.tight_layout()
plt.show()

## Top hits per lineage by padj_treatment

In [ ]:
TOP_N = 25
for ln in LINEAGES:
    df = metrics[ln].sort_values("padj_treatment").head(TOP_N)
    print(f"\n=== TOP {TOP_N} by padj_treatment -- {ln} ===")
    cols = [
        "gene_id",
        "n_features_post_filter",
        "padj_treatment",
        "pvalue_treatment",
        "delta_psi_treatment",
        "delta_entropy",
        "entropy_padj_bh",
        "pattern",
    ]
    avail = [c for c in cols if c in df.columns]
    with pd.option_context(
        "display.float_format", lambda v: f"{v:.3e}" if abs(v) < 1e-2 else f"{v:.4f}"
    ):
        print(df[avail].to_string(index=False))

## Top hits by |delta_psi_treatment| (conditioning on padj<0.05)

In [ ]:
for ln in LINEAGES:
    df = metrics[ln].copy()
    sig = df[df["padj_treatment"] < PADJ_THRESHOLD].copy()
    sig["abs_dpsi"] = sig["delta_psi_treatment"].abs()
    sig = sig.sort_values("abs_dpsi", ascending=False).head(TOP_N)
    print(f"\n=== TOP {TOP_N} by |delta_psi_treatment| (padj<{PADJ_THRESHOLD}) -- {ln} ===")
    cols = [
        "gene_id",
        "n_features_post_filter",
        "padj_treatment",
        "delta_psi_treatment",
        "delta_entropy",
        "entropy_padj_bh",
        "pattern",
    ]
    avail = [c for c in cols if c in sig.columns]
    with pd.option_context(
        "display.float_format", lambda v: f"{v:.3e}" if abs(v) < 1e-2 else f"{v:.4f}"
    ):
        print(sig[avail].to_string(index=False))

## Top hits by |delta_entropy| (supplementary)

In [ ]:
for ln in LINEAGES:
    df = metrics[ln].copy()
    df["abs_dH"] = df["delta_entropy"].abs()
    df = df.sort_values("abs_dH", ascending=False).head(TOP_N)
    print(f"\n=== TOP {TOP_N} by |delta_entropy| (supplementary) -- {ln} ===")
    cols = [
        "gene_id",
        "n_features_post_filter",
        "padj_treatment",
        "delta_psi_treatment",
        "H_induced_True",
        "H_induced_False",
        "delta_entropy",
        "entropy_padj_bh",
        "pattern",
    ]
    avail = [c for c in cols if c in df.columns]
    with pd.option_context(
        "display.float_format", lambda v: f"{v:.3e}" if abs(v) < 1e-2 else f"{v:.4f}"
    ):
        print(df[avail].to_string(index=False))